Import required libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)

Prepare a tiny English-French Dataset

In [2]:
pairs = [
    ("i love cats", "j aime les chats"),
    ("i love dogs", "j aime les chiens"),
    ("i like cats", "j aime les chats"),
    ("i like dogs", "j aime les chiens"),
    ("i see cats", "je vois les chats"),
    ("i see dogs", "je vois les chiens"),
    ("you love cats", "tu aimes les chats"),
    ("you love dogs", "tu aimes les chiens"),
    ("you like cats", "tu aimes les chats"),
    ("you like dogs", "tu aimes les chiens"),
]


Build a vocabulary from the dataset

In [3]:
def build_vocab(sentences):

    vocab = {
        "<PAD>": 0,
        "<SOS>": 1,
        "<EOS>": 2,
        "<UNK>": 3
    }

    for sentence in sentences:

        for word in sentence.split():

            if word not in vocab:
                vocab[word] = len(vocab)

    return vocab


src_sentences = [src for src, tgt in pairs]
tgt_sentences = [tgt for src, tgt in pairs]

src_vocab = build_vocab(src_sentences)
tgt_vocab = build_vocab(tgt_sentences)

src_vocab_size = len(src_vocab)
tgt_vocab_size = len(tgt_vocab)

print("Source vocabulary:")
print(src_vocab)

print("\nTarget vocabulary:")
print(tgt_vocab)

Source vocabulary:
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'i': 4, 'love': 5, 'cats': 6, 'dogs': 7, 'like': 8, 'see': 9, 'you': 10}

Target vocabulary:
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3, 'j': 4, 'aime': 5, 'les': 6, 'chats': 7, 'chiens': 8, 'je': 9, 'vois': 10, 'tu': 11, 'aimes': 12}


Tokenize the sentences, from the token ids created through the vocabulary

In [5]:
def tokenize(sentence, vocab):

    tokens = ["<SOS>"]

    for word in sentence.split():
        tokens.append(word)

    tokens.append("<EOS>")

    ids = [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

    return ids


# Example
print("\nExample tokenization:")
print(tokenize("i love cats", src_vocab))


Example tokenization:
[1, 4, 5, 6, 2]


Create an encoder with context vector being created using vector representation

In [6]:
class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )


    def forward(self, x):

        # x:
        # [batch, sequence_length]

        embedded = self.embedding(x)

        # embedded:
        # [batch, sequence_length, embedding_dim]

        outputs, (hidden, cell) = self.lstm(
            embedded
        )

        return hidden, cell

Create the decoder architecture

In [7]:
class Decoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            hidden_dim,
            vocab_size
        )


    def forward(
        self,
        x,
        hidden,
        cell
    ):

        # x:
        # [batch, 1]

        embedded = self.embedding(x)

        # embedded:
        # [batch, 1, embedding_dim]

        output, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        # output:
        # [batch, 1, hidden_dim]

        prediction = self.output_layer(
            output
        )

        # prediction:
        # [batch, 1, vocab_size]

        return prediction, hidden, cell

Create a seq2seq model representation that puts together the entire architecture of the Encoder, decoder, forward linear layer and the softmax for the next autoregressive token probability.

In [9]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder
    ):

        super().__init__()

        self.encoder = encoder
        self.decoder = decoder


    def forward(
        self,
        source,
        target,
        teacher_forcing_ratio=0.5
    ):

        batch_size = source.shape[0]

        target_length = target.shape[1]

        target_vocab_size = self.decoder.output_layer.out_features

        # Store predictions
        outputs = torch.zeros(
            batch_size,
            target_length,
            target_vocab_size,
            device=source.device
        )

        # ----------------------------------------------------
        # ENCODER
        # ----------------------------------------------------

        hidden, cell = self.encoder(source)

        # ----------------------------------------------------
        # First decoder input = <SOS>
        # ----------------------------------------------------

        decoder_input = target[:, 0].unsqueeze(1)

        # ----------------------------------------------------
        # DECODER
        # ----------------------------------------------------

        for t in range(1, target_length):

            prediction, hidden, cell = self.decoder(
                decoder_input,
                hidden,
                cell
            )

            outputs[:, t:t+1] = prediction

            # Decide whether to use:
            #
            # 1. Correct target token
            # OR
            # 2. Model prediction

            teacher_force = (
                torch.rand(1).item()
                < teacher_forcing_ratio
            )

            predicted_token = prediction.argmax(
                dim=2
            )

            if teacher_force:

                decoder_input = target[:, t].unsqueeze(1)

            else:

                decoder_input = predicted_token

        return outputs

Create an instance of the classes for the architecture parts (encoder, decoder, seq2seq architecture)

In [10]:
embedding_dim = 64
hidden_dim = 128

encoder = Encoder(
    src_vocab_size,
    embedding_dim,
    hidden_dim
)

decoder = Decoder(
    tgt_vocab_size,
    embedding_dim,
    hidden_dim
)

model = Seq2Seq(
    encoder,
    decoder
)

Create an instance of the optimizer and loss function

In [11]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    ignore_index=tgt_vocab["<PAD>"]
)




Prepare data for training

In [12]:
source_data = []
target_data = []

for src, tgt in pairs:

    source_data.append(
        tokenize(src, src_vocab)
    )

    target_data.append(
        tokenize(tgt, tgt_vocab)
    )


source_tensor = torch.tensor(
    source_data,
    dtype=torch.long
)

target_tensor = torch.tensor(
    target_data,
    dtype=torch.long
)


print("\nSource shape:")
print(source_tensor.shape)

print("\nTarget shape:")
print(target_tensor.shape)


Source shape:
torch.Size([10, 5])

Target shape:
torch.Size([10, 6])


Train the model over 1000 epochs

In [13]:
epochs = 1000

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    output = model(
        source_tensor,
        target_tensor,
        teacher_forcing_ratio=0.5
    )

    # --------------------------------------------------------
    # Ignore prediction at position 0 because it is <SOS>
    # --------------------------------------------------------

    output = output[:, 1:, :]

    target = target_tensor[:, 1:]

    # --------------------------------------------------------
    # CrossEntropyLoss expects:
    #
    # predictions:
    # [batch, vocab_size, sequence]
    #
    # targets:
    # [batch, sequence]
    # --------------------------------------------------------

    output = output.permute(
        0, 2, 1
    )

    loss = criterion(
        output,
        target
    )

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 100 == 0:

        print(
            f"Epoch {epoch + 1}/{epochs} "
            f"| Loss: {loss.item():.4f}"
        )

Epoch 100/1000 | Loss: 0.0217
Epoch 200/1000 | Loss: 0.0058
Epoch 300/1000 | Loss: 0.0029
Epoch 400/1000 | Loss: 0.0018
Epoch 500/1000 | Loss: 0.0013
Epoch 600/1000 | Loss: 0.0009
Epoch 700/1000 | Loss: 0.0007
Epoch 800/1000 | Loss: 0.0006
Epoch 900/1000 | Loss: 0.0005
Epoch 1000/1000 | Loss: 0.0004
